In [2]:
import pandas as pd

In [3]:
df_gaps = pd.read_csv('mbj_gaps_binary_2025_0425_pmg_info.csv')
df_groups = pd.read_json('binary-group-df.json')

In [16]:
df_gaps[df_gaps.formula =='PbSe']

,Unnamed: 0,formula,material_id,direct,transition,band_gap
225,84,PbSe,mp-2201,True,L-L,0.9620
231,62,PbSe,wbm-53023,True,L-L,1.0009
232,19,PbSe,wbm-53024,True,L-L,1.0032
233,73,PbSe,wbm-53019,True,L-L,1.0034
234,45,PbSe,wbm-53026,True,X-X,1.0043


In [5]:
mbj_gap_dict = {row.material_id: [row.band_gap, row.direct] for _, row in df_gaps.iterrows()}

In [6]:
new_rows = []
for index, row in df_groups.iterrows():
    gaps_hse06 = [mbj_gap_dict.get(material_id) for material_id in row.mp_ids]
    is_valid = [x is not None for x in gaps_hse06]
    outdict = {}
    for key in ['A_elements', 'compositions', 'X_element', 'mp_ids', 'band_gaps']:
        value = getattr(row, key)
        outdict[key] = [x for mask, x in zip(is_valid, value) if mask is True]
    outdict['gaps_mbj'] = [x for mask, x in zip(is_valid, gaps_hse06) if mask is True]
    if len(outdict['mp_ids']) > 1:
        new_rows.append(outdict)

In [7]:
df_mbj = pd.DataFrame.from_records(new_rows)
df_mbj

,A_elements,compositions,X_element,mp_ids,band_gaps,gaps_mbj
0,"[[Si], [Si], [Si], [Si], [Si], [Si], [Si], [Si]]","[HfSi, ZrSi, CaSi, HfSi, SrSi, SrSi, ZrSi, HfSi]","[Hf, Zr, Ca, Hf, Sr, Sr, Zr, Hf]","[wbm-54949, wbm-110377, wbm-185205, wbm-185767...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]","[[0.0, False], [0.0, False], [0.0, False], [0...."
1,"[[O], [O], [O], [O], [O], [O], [O], [O]]","[IrO2, PbO2, PdO2, MoO2, IrO2, RhO2, SnO2, PdO2]","[Ir, Pb, Pd, Mo, Ir, Rh, Sn, Pd]","[wbm-37084, wbm-37489, wbm-37528, mp-510536, m...","[0.0, 0.12, 0.22, 0.0, 0.0, 0.0, 0.6519, 0.0]","[[0.0, False], [0.0, False], [0.87559999999999..."
2,"[[N], [N], [N], [N], [N], [N], [N], [N], [N]]","[ZrN, HfN, YN, ScN, ZrN, ScN, HfN, YN, ZrN]","[Zr, Hf, Y, Sc, Zr, Sc, Hf, Y, Zr]","[wbm-32432, wbm-159764, wbm-160309, wbm-216647...","[0.0, 0.0, 0.14, 0.0, 0.0, 0.0, 0.0, 0.151, 0.0]","[[0.0, False], [0.0, False], [1.141, False], [..."
3,"[[O], [O], [O]]","[IrO2, IrO2, IrO2]","[Ir, Ir, Ir]","[wbm-37081, wbm-37082, wbm-165143]","[0.0, 0.0, 0.0]","[[0.0, False], [0.0, False], [0.0, False]]"
4,"[[Se], [Se], [Se], [Se], [Se], [Se], [Se], [Se...","[GeSe, GeSe, GeSe, PbSe, PbSe, PbSe, PbSe, ZnS...","[Ge, Ge, Ge, Pb, Pb, Pb, Pb, Zn, Sn, Pb, Ce]","[wbm-20072, wbm-20073, wbm-20074, wbm-53019, w...","[0.84, 0.48, 0.07, 0.99, 0.99, 0.8, 0.8, 0.0, ...","[[0.2382999999999997, True], [0.68540000000000..."
...,...,...,...,...,...,...
97,"[[Sb], [Sb]]","[Nb3Sb, Ta3Sb]","[Nb, Ta]","[mp-2053, mp-541]","[0.0, 0.0]","[[0.0, False], [0.0, False]]"
98,"[[Sn], [Sn]]","[SnAs3, SnP3]","[As, P]","[wbm-3411, mp-7541]","[0.0, 0.0]","[[0.0, False], [0.0, False]]"
99,"[[Sc], [Sc]]","[ScP, ScAs]","[P, As]","[wbm-46303, wbm-51467]","[0.0, 0.0]","[[0.0, False], [0.0, False]]"
100,"[[As], [As]]","[Rb3As, K3As]","[Rb, K]","[mp-7898, mp-14018]","[0.0, 0.11180000000000001]","[[1.1772999999999998, False], [1.5392, False]]"


In [8]:
def gaps_valid(gaps):
    """Check if the gaps are valid"""
    has_direct = any(entry[0] < 1.5 and entry[0] > 0.15 for entry in gaps)  #  There is a direct but small gap betweem 0.1 and 0.5
    has_low = any(entry[0] < 0.15 for entry in gaps)
    return has_direct and has_low

In [9]:
valid = df_mbj.loc[df_mbj.gaps_mbj.apply(gaps_valid)]
valid

,A_elements,compositions,X_element,mp_ids,band_gaps,gaps_mbj
1,"[[O], [O], [O], [O], [O], [O], [O], [O]]","[IrO2, PbO2, PdO2, MoO2, IrO2, RhO2, SnO2, PdO2]","[Ir, Pb, Pd, Mo, Ir, Rh, Sn, Pd]","[wbm-37084, wbm-37489, wbm-37528, mp-510536, m...","[0.0, 0.12, 0.22, 0.0, 0.0, 0.0, 0.6519, 0.0]","[[0.0, False], [0.0, False], [0.87559999999999..."
2,"[[N], [N], [N], [N], [N], [N], [N], [N], [N]]","[ZrN, HfN, YN, ScN, ZrN, ScN, HfN, YN, ZrN]","[Zr, Hf, Y, Sc, Zr, Sc, Hf, Y, Zr]","[wbm-32432, wbm-159764, wbm-160309, wbm-216647...","[0.0, 0.0, 0.14, 0.0, 0.0, 0.0, 0.0, 0.151, 0.0]","[[0.0, False], [0.0, False], [1.141, False], [..."
4,"[[Se], [Se], [Se], [Se], [Se], [Se], [Se], [Se...","[GeSe, GeSe, GeSe, PbSe, PbSe, PbSe, PbSe, ZnS...","[Ge, Ge, Ge, Pb, Pb, Pb, Pb, Zn, Sn, Pb, Ce]","[wbm-20072, wbm-20073, wbm-20074, wbm-53019, w...","[0.84, 0.48, 0.07, 0.99, 0.99, 0.8, 0.8, 0.0, ...","[[0.2382999999999997, True], [0.68540000000000..."
7,"[[Te], [Te], [Te], [Te], [Te], [Te], [Te], [Te]]","[GeTe, SnTe, SnTe, SnTe, GeTe, CeTe, PbTe, SnTe]","[Ge, Sn, Sn, Sn, Ge, Ce, Pb, Sn]","[wbm-20160, wbm-56490, wbm-56491, wbm-56493, m...","[0.36, 0.02, 0.22, 0.02, 0.8086, 0.0, 0.8063, ...","[[0.3582999999999998, True], [0.29830000000000..."
8,"[[S], [S], [S], [S], [S], [S], [S], [S], [S], ...","[CdS, CdS, PbS, PbS, ScS, SnS, ZnS, ZnS, PbS, ...","[Cd, Cd, Pb, Pb, Sc, Sn, Zn, Zn, Pb, Ce, Sc]","[wbm-50441, wbm-50442, wbm-50757, wbm-50759, w...","[0.28, 0.28, 0.45, 0.8200000000000001, 0.0, 0....","[[1.7102, False], [1.7102, False], [1.20450000..."
9,"[[Sc], [Sc], [Sc], [Sc], [Sc], [Sc], [Sc], [Sc]]","[ScS, ScSb, ScN, ScP, ScN, ScS, ScSb, ScAs]","[S, Sb, N, P, N, S, Sb, As]","[wbm-50821, wbm-181982, wbm-216647, mp-2807, m...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]","[[0.0, False], [0.0, False], [0.96939999999999..."
12,"[[Ce], [Ce], [Ce], [Ce], [Ce], [Ce], [Ce], [Ce]]","[CeBi, CeP, CeO, CeTe, CeS, CeSb, CeSe, CeAs]","[Bi, P, O, Te, S, Sb, Se, As]","[mp-23285, mp-2154, mp-10688, mp-1525, mp-1096...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]","[[0.2061000000000001, False], [0.0, False], [0..."
17,"[[Se], [Se], [Se], [Se]]","[ScSe, PbSe, ZrSe, ScSe]","[Sc, Pb, Zr, Sc]","[wbm-52086, wbm-53022, wbm-53144, wbm-182004]","[0.0, 0.1, 0.0, 0.0]","[[0.0, False], [0.4577999999999997, True], [0...."
19,"[[O], [O], [O], [O], [O], [O], [O], [O]]","[CdO, CdO, CdO, ZnO, ZnO, ZnO, CeO, CdO]","[Cd, Cd, Cd, Zn, Zn, Zn, Ce, Cd]","[wbm-43907, wbm-43908, wbm-43912, wbm-44816, w...","[0.0, 0.0, 0.0, 0.74, 0.72, 0.71, 0.0, 0.0]","[[1.3151000000000002, False], [1.3159, False],..."
20,"[[Sb], [Sb], [Sb], [Sb], [Sb], [Sb], [Sb]]","[YSb, ScSb, YSb, BiSb, CeSb, YSb, ScSb]","[Y, Sc, Y, Bi, Ce, Y, Sc]","[wbm-60018, wbm-181982, wbm-191113, mp-1227290...","[0.0, 0.0, 0.0, 0.1101, 0.0, 0.0, 0.0]","[[0.0, False], [0.0, False], [0.0, False], [0...."


In [10]:
combinations = []
for key, row in valid.iterrows():
    mbj_gaps = row.gaps_mbj
    for i in range(len(row.compositions)):
        for j in range(i, len(row.compositions)):
            if any([mbj_gaps[i][0] < 0.3, mbj_gaps[j][0] < 0.3]) and \
            any([mbj_gaps[i][0] > 0.2, mbj_gaps[j][0] > 0.2]) and \
            not any([mbj_gaps[i][0] > 0.8, mbj_gaps[j][0] > 0.8]):
            # if (mbj_gaps[i][0] < 0.1 and mbj_gaps[j][0] < 0.6 and mbj_gaps[j][0] > 0.1 ) or (mbj_gaps[j][0] < 0.1 and mbj_gaps[i][0] < 0.6 and mbj_gaps[i][0] > 0.1 ):
                # if not mbj_gaps[i][1]:
                #     j, i = i, j
                data = {'comp_a': row.compositions[i], 
                        'comp_b': row.compositions[j],
                        'gap_a': mbj_gaps[i][0],
                        'gap_b': mbj_gaps[j][0],
                        'mp_id_a': row.mp_ids[i],
                        'mp_id_b': row.mp_ids[j],
                        'gap_direct_a':mbj_gaps[i][1],
                        'gap_direct_b': mbj_gaps[j][1],
                       }
                if data['comp_a'] == data['comp_b']:
                    continue
                combinations.append(data)

In [11]:
pairs = pd.DataFrame.from_dict(combinations)
#pairs = pairs[pairs.gap_direct_a | pairs.gap_direct_b]

In [12]:
pairs

,comp_a,comp_b,gap_a,gap_b,mp_id_a,mp_id_b,gap_direct_a,gap_direct_b
0,GeSe,ZnSe,0.2383,0.4446,wbm-20072,wbm-61175,True,False
1,GeSe,SnSe,0.2383,0.1129,wbm-20072,wbm-228459,True,False
2,GeSe,CeSe,0.2383,0.0000,wbm-20072,mp-2563,True,False
3,GeSe,SnSe,0.6854,0.1129,wbm-20073,wbm-228459,True,False
4,GeSe,CeSe,0.6854,0.0000,wbm-20073,mp-2563,True,False
5,GeSe,ZnSe,0.0000,0.4446,wbm-20074,wbm-61175,False,False
6,ZnSe,SnSe,0.4446,0.1129,wbm-61175,wbm-228459,False,False
7,ZnSe,CeSe,0.4446,0.0000,wbm-61175,mp-2563,False,False
8,GeTe,SnTe,0.3583,0.2983,wbm-20160,wbm-56490,True,True
9,GeTe,CeTe,0.3583,0.0000,wbm-20160,mp-1525,True,False


In [13]:
 pairs[pairs.gap_direct_a & pairs.gap_direct_b]

,comp_a,comp_b,gap_a,gap_b,mp_id_a,mp_id_b,gap_direct_a,gap_direct_b
8,GeTe,SnTe,0.3583,0.2983,wbm-20160,wbm-56490,True,True
10,GeTe,SnTe,0.3583,0.2569,wbm-20160,mp-1883,True,True
50,GaSb,InSb,0.7962,0.0973,mp-1156,mp-20012,True,True


In [13]:
pairs.to_csv('mbj_result_binary_20250426.csv')